# TT-rank truncation による Linear output error と norm 上界
## 理論・PyTorch 実装・数値検証

配置想定：

```text
notebooks/30_tt_mps/10_fashion_mnist_mlp/
├── 00_mlp_ttlinear_logits_equivalence.ipynb
└── 01_ttlinear_output_error_norm_bounds.ipynb
```

## 到達目標

1. TT-rank truncation により $\widetilde W$ を作る
2. weight error $\Delta W=\widetilde W-W$ を定義する
3. output error $\Delta Y=\widetilde Y-Y=X\Delta W^{\mathsf T}$ を確認する
4. Frobenius norm と spectral norm の違いを確認する
5. $\|\Delta Y\|_F\le\|X\|_F\|\Delta W\|_2$ を数値検証する
6. $\|\Delta Y\|_F\le\|X\|_2\|\Delta W\|_F$ を数値検証する

今回は **Linear layer 単体の truncation error と norm upper bound** に限定します。

扱わないもの：ReLU、hidden activation error、logits error、classification margin、prediction、accuracy、Fashion-MNIST、fine-tuning、学習ループ、optimizer、backpropagation。


In [25]:
import torch
import torch.nn as nn
from torch.testing import assert_close

torch.manual_seed(0)

dtype = torch.float64
device = torch.device("cpu")

rtol = 1e-10
atol = 1e-10
inequality_atol = 1e-12

batch_size = 5
n1, n2 = 3, 4
m1, m2 = 4, 5
input_dim = n1 * n2
output_dim = m1 * m2

tt_rank_truncated = 2

print(f"batch_size         = {batch_size}")
print(f"input_dim          = {input_dim}")
print(f"output_dim         = {output_dim}")
print(f"TT truncation rank = {tt_rank_truncated}")


batch_size         = 5
input_dim          = 12
output_dim         = 20
TT truncation rank = 2


## 1. 問題設定と shape

$$
X\in\mathbb{R}^{B\times n},\qquad
W\in\mathbb{R}^{h\times n},\qquad
b\in\mathbb{R}^{h}.
$$

dense Linear は、

$$
Y=XW^{\mathsf T}+b\in\mathbb{R}^{B\times h}.
$$

TT-rank truncation 後の近似 weight を $\widetilde W\in\mathbb{R}^{h\times n}$ とし、

$$
\widetilde Y=X\widetilde W^{\mathsf T}+b.
$$

weight error と output error を、

$$
\boxed{\Delta W=\widetilde W-W},
$$

$$
\boxed{\Delta Y=\widetilde Y-Y}
$$

と定義します。

bias は両者で同じなので、

$$
\begin{aligned}
\Delta Y
&=(X\widetilde W^{\mathsf T}+b)-(XW^{\mathsf T}+b)\\
&=X(\widetilde W^{\mathsf T}-W^{\mathsf T})\\
&=X(\widetilde W-W)^{\mathsf T}\\
&=\boxed{X\Delta W^{\mathsf T}}.
\end{aligned}
$$

shape は、

$$
X\Delta W^{\mathsf T}:(B\times n)(n\times h)=B\times h.
$$

今回の具体値は、

$$
X:(5,12),\quad W,\widetilde W,\Delta W:(20,12),\quad Y,\widetilde Y,\Delta Y:(5,20).
$$


## 2. Frobenius norm と spectral norm

### Frobenius norm

$$
\boxed{\|A\|_F=\sqrt{\sum_{i,j}A_{ij}^2}}
$$

SVD を $A=U\Sigma V^{\mathsf T}$、特異値を $\sigma_1\ge\sigma_2\ge\cdots\ge0$ とすると、

$$
\boxed{\|A\|_F=\sqrt{\sum_k\sigma_k^2}}.
$$

### Spectral norm

$$
\boxed{
\|A\|_2
=
\sup_{v\ne0}\frac{\|Av\|_2}{\|v\|_2}
=
\sigma_{\max}(A)
}
$$

つまり「行列 $A$ がベクトルを最も大きく伸ばす倍率」です。

$$
\boxed{\|A\|_2\le\|A\|_F}
$$

も成り立ちます。

| 数学 | PyTorch |
|---|---|
| vector 2-norm $\|v\|_2$ | `torch.linalg.vector_norm(v, ord=2)` |
| matrix Frobenius norm $\|A\|_F$ | `torch.linalg.matrix_norm(A, ord="fro")` |
| matrix spectral norm $\|A\|_2$ | `torch.linalg.matrix_norm(A, ord=2)` |

`torch.linalg.matrix_norm(A, ord=2)` は Frobenius norm ではなく、最大特異値です。


## 3. 二つの norm 上界

一般に、

$$
\|AB\|_F\le\|A\|_F\|B\|_2,
$$

$$
\|AB\|_F\le\|A\|_2\|B\|_F.
$$

今回、$A=X$、$B=\Delta W^{\mathsf T}$ と置きます。

転置で Frobenius norm と spectral norm は変わらないので、

$$
\|\Delta W^{\mathsf T}\|_2=\|\Delta W\|_2,
\qquad
\|\Delta W^{\mathsf T}\|_F=\|\Delta W\|_F.
$$

よって、

$$
\boxed{
\|\Delta Y\|_F
=\|X\Delta W^{\mathsf T}\|_F
\le\|X\|_F\|\Delta W\|_2
}
$$

および、

$$
\boxed{
\|\Delta Y\|_F
=\|X\Delta W^{\mathsf T}\|_F
\le\|X\|_2\|\Delta W\|_F
}
$$

を得ます。


## 4. 演習1 — dense weight の TT-matrix tensorization

元の weight は、

$$
W\in\mathbb{R}^{(m_1m_2)\times(n_1n_2)}.
$$

これを paired tensor、

$$
\mathcal W\in\mathbb{R}^{m_1\times n_1\times m_2\times n_2}
$$

として見ます。

対応は、

$$
\boxed{
\mathcal W[i_1,j_1,i_2,j_2]
=
W[i_1m_2+i_2,\;j_1n_2+j_2]
}
$$

です。

ヒント：まず $(m_1,m_2,n_1,n_2)$ に reshape し、その後 site ごとの pair が隣り合うよう axis を並べ替えます。


In [26]:
# TODO 1:
def dense_weight_to_paired_tensor(
    weight: torch.Tensor,
    out_modes: tuple[int, int],
    in_modes: tuple[int, int],
) -> torch.Tensor:
    """2-core TT-matrix 用に dense weight を paired tensor へ変形する。"""
#
    # 実装条件:
    # - weight は2次元
    # - weight.shape == (m1*m2, n1*n2)
    # - 不正入力は ValueError
    # - 出力 shape は (m1, n1, m2, n2)
    # - in-place 演算は禁止
        # TODO 9:
# dense_to_tt_matrix_tensor を実装してください。
    """dense weight W を TT-SVD 入力用の combined tensor へ並べ替える。

    処理の流れ:
        1. W.shape == (m, n) を検証（m,n = product(out/in_modes)）
        2. grouped: (m_1,...,m_d, n_1,...,n_d) へ reshape
        3. grouped → interleaved: (m_1,n_1,...,m_d,n_d) へ permute
    引数:
        W: dense weight, shape (m, n)
            dtype float64, device CPU（入力は変更しない）
        out_modes: 行側 modes (m_1, ..., m_d)
        in_modes: 列側 modes (n_1, ..., n_d)
        verbose: True なら grouped / perm / interleaved / combined を表示

    戻り値:
        A: combined tensor, shape (q_1, ..., q_d)
            q_k = m_k * n_k

    注意:
        TT-SVD 自体はこの関数では行わない（並べ替えのみ）。
    """

    # list / tuple の違いをここでなくす。
    # 以後は shape と比較しやすい tuple として扱う。
    out_modes = tuple(out_modes)
    in_modes = tuple(in_modes)

    # =========================================================
    # 1. validate
    # =========================================================

    # site 数 d を決める。
    # 例:
    # out_modes = (2, 3, 4) なら d = 3
    d = len(out_modes)

    # W は行列である必要がある。
    assert weight.ndim == 2,(
        print(f"W must be 2D, got shape={tuple(weight.shape)}")
    )
    assert len(out_modes) == len(in_modes)
    assert len(out_modes) > 0

    assert all(mode > 0 for mode in out_modes)
    assert all(mode > 0 for mode in in_modes)

    assert weight.dtype == torch.float64
    # assert W.device.type == "cpu"
    # 出力側と入力側で site 数が一致する必要がある。
    # 例:
    # out_modes = (2, 3, 4)
    # in_modes  = (5, 6, 7)
    # → OK
    #
    # out_modes = (2, 3)
    # in_modes  = (4, 5, 6)
    # → NG
    assert len(out_modes) == len(in_modes),(
        print(f"len(out_modes)={len(out_modes)}, len(in_modes)={len(in_modes)}")
    )

    # 空の mode list は許可しない。
    # d=0 では TT-SVD 用 tensor を作る意味がない。
    assert d != 0,(
        print("d=0")
    )

    # 各 mode は正の整数である必要がある。
    # 例:
    # (2, 3, 4) → OK
    # (2, 0, 4) → NG
    # (2, -3, 4) → NG
    assert all(mode > 0 for mode in out_modes), (
        f"out_modes の各要素は正の整数である必要があります: {out_modes}"
    )
    assert all(mode > 0 for mode in in_modes), (
        f"in_modes の各要素は正の整数である必要があります: {in_modes}"
    )

    # dense W の実際の shape を取得する。
    # m: output feature 数
    # n: input feature 数
    m, n = weight.shape

    # modes が要求する dense shape を作る。
    # expected_m = m1 * m2 * ... * md
    # expected_n = n1 * n2 * ... * nd
    expected_m = 1
    for mode in out_modes:
        expected_m = mode * expected_m
    expected_n = 1
    for mode in in_modes:
        expected_n = mode * expected_n

    # W.shape が modes と一致するかを確認する。
    # 例:
    # W.shape = (6, 20)
    # out_modes = (2, 3)  -> expected_m = 6
    # in_modes  = (4, 5)  -> expected_n = 20
    # → OK
    # W.shape = (7, 20) なら NG
    assert expected_m == m,(
        print(f"miss match shape,expected_m={expected_m },m={m}")
    )
    assert expected_n == n,(
        print(f"miss match shape,expected_n={expected_n },m={n}")
    ) 

    # 2. grouped
    # W の row axis を out_modes に、
    # W の column axis を in_modes に分解する reshape を書く。
    grouped = weight.reshape(*out_modes , *in_modes)

    # 3. interleaved
    # grouped の axis:
    # (m1, ..., md, n1, ..., nd)
    #
    # 欲しい axis:
    # (m1, n1, ..., md, nd)
    #
    # perm を loop で構築する。
    perm = []

    for k in range(d):
        # output mode m_{k+1} の元 axis を追加
        perm.append(k)
        # input mode n_{k+1} の元 axis を追加
        perm.append(d + k)

    return grouped.permute(*perm)



## 5. 演習2 — rank truncation 付き 2-core TT-SVD

paired tensor を、

$$
M\in\mathbb{R}^{(m_1n_1)\times(m_2n_2)}
$$

へ unfold します。

今回の shape は、

$$
M\in\mathbb{R}^{12\times20}.
$$

SVD、

$$
M=U\Sigma V^{\mathsf T}
$$

を計算し、上位 $r=2$ 個だけを残します。

core shape は、

$$
G^{(1)}\in\mathbb{R}^{1\times m_1\times n_1\times r},
$$

$$
G^{(2)}\in\mathbb{R}^{r\times m_2\times n_2\times1}.
$$

今回は意図的に rank を切るので、$\widetilde W\neq W$ が生じます。


In [27]:
# TODO 2:
def tt_svd_matrix_2core_with_rank(
    weight: torch.Tensor,
    out_modes: tuple[int, int],
    in_modes: tuple[int, int],
    rank: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """dense weight を 2-core TT-matrix へ rank truncation 付きで分解する。"""
    #
    # 手順:
    # 1. TODO 1 の paired tensor を作る
    paired_weight = dense_weight_to_paired_tensor(weight,out_modes,in_modes)
    
    # 2. 先頭 site と残りに unfold
    # d=2 なら (m1*n1, m2*n2)。d が増えても同じ式。
    q_modes = [m_k * n_k for m_k, n_k in zip(out_modes, in_modes)]
    paired_weight = paired_weight.reshape(q_modes[0], -1)

    # rank は 1 以上、かつ特異値の個数以下
    max_rank = min(paired_weight.shape)
    if (
        not isinstance(rank, int)
        or isinstance(rank, bool)
        or not 1 <= rank <= max_rank
    ):
        raise ValueError(
            f"rank は 1 以上 min(M.shape)={max_rank} 以下の整数である必要があります: {rank}"
        )

    # 3. torch.linalg.svd(..., full_matrices=False)
    U,S,Vh = torch.linalg.svd(paired_weight, full_matrices=False)

    # 4. 上位 rank 個だけ保持
    U_paired = U[:,:rank]
    S_paired = S[:rank]
    Vh_paired = Vh[:rank,:]

    # 5. core_1: (1, m1, n1, rank)
    core_1 = U_paired
    core_1 = core_1.reshape(1, m1, n1, rank)

    # 6. core_2: (rank, m2, n2, 1)
    core_2 = torch.diag(S_paired)@Vh_paired
    core_2 = core_2.reshape(rank, m2, n2, 1)

    return core_1, core_2


## 6. 演習3 — TT-matrix の dense reconstruction

これは forward のためではなく **検証専用**です。

$$
\widetilde{\mathcal W}_{i_1,j_1,i_2,j_2}
=
\sum_{\alpha=1}^{r}
G^{(1)}_{1,i_1,j_1,\alpha}
G^{(2)}_{\alpha,i_2,j_2,1}.
$$

paired tensor を復元したあと、axis を grouped order に戻し、

$$
(m_1,m_2,n_1,n_2)\rightarrow(m_1m_2,n_1n_2)
$$

とします。

境界 TT rank は $r_0=r_2=1$ です。数学では唯一の境界添字を $1$、Python では size 1 軸の唯一の index を `0` とします。


In [28]:
from collections.abc import Sequence
import math


def reconstruct_tt_matrix(cores: Sequence[torch.Tensor]) -> torch.Tensor:
    """任意個数の TT-matrix core から dense weight を復元する。検証専用。

    各 core は (r_{k-1}, m_k, n_k, r_k)。
    m_k / n_k は core.shape から読むので、m1 などの名前は不要。
    戻り値 shape は (prod(m_k), prod(n_k))。
    """
    if len(cores) < 1:
        raise ValueError("cores は1個以上必要です")

    for k, core in enumerate(cores):
        if core.ndim != 4:
            raise ValueError(
                f"core[{k}] は4階である必要があります: shape={tuple(core.shape)}"
            )

    if cores[0].shape[0] != 1 or cores[-1].shape[-1] != 1:
        raise ValueError(
            "境界 rank は1である必要があります: "
            f"left={cores[0].shape[0]}, right={cores[-1].shape[-1]}"
        )

    for k, (left, right) in enumerate(zip(cores, cores[1:])):
        if left.shape[-1] != right.shape[0]:
            raise ValueError(
                f"core[{k}] と core[{k + 1}] の bond が一致しません: "
                f"{left.shape[-1]} != {right.shape[0]}"
            )

    out_modes = tuple(int(core.shape[1]) for core in cores)
    in_modes = tuple(int(core.shape[2]) for core in cores)
    d = len(cores)

    current = cores[0].squeeze(0)  # (m1, n1, r1)
    for core in cores[1:]:
        current = torch.tensordot(current, core, dims=([-1], [0]))
    current = current.squeeze(-1)  # (m1, n1, ..., md, nd)

    perm = [2 * k for k in range(d)] + [2 * k + 1 for k in range(d)]
    grouped = current.permute(*perm)  # (m1, ..., md, n1, ..., nd)
    return grouped.reshape(math.prod(out_modes), math.prod(in_modes))


In [29]:
# TODO 3:
def reconstruct_tt_matrix_2core(
    core_1: torch.Tensor,
    core_2: torch.Tensor,
) -> torch.Tensor:
    """2-core TT-matrix を dense weight へ復元する。検証専用。"""
#
    # 必須:
    # - core_1 / core_2 は4次元
    if core_1.ndim != 4:
        raise ValueError(f"core_1 は4階である必要があります: shape={tuple(core_1.shape)}")
    if core_2.ndim != 4:
        raise ValueError(f"core_2 は4階である必要があります: shape={tuple(core_2.shape)}")
    # - 左右 boundary rank は1
    if core_1.shape[0] != 1:
        raise ValueError(f"core_1 の左 boundary rank は1である必要があります: shape={tuple(core_1.shape)}")
    if core_2.shape[-1] != 1:
        raise ValueError(f"core_2 の右 boundary rank は1である必要があります: shape={tuple(core_2.shape)}")
    # - internal rank は一致
    if core_1.shape[-1] != core_2.shape[0]:
        raise ValueError(
            f"internal rank が一致しません: core_1.shape[-1]={core_1.shape[-1]}, core_2.shape[0]={core_2.shape[0]}"
        )

    _, m1, n1, _ = core_1.shape
    _, m2, n2, _ = core_2.shape

    # - paired tensor の shape は (m1, n1, m2, n2)
    # - 最終 dense weight は (m1*m2, n1*n2)
    paired = torch.einsum("abcd,defg->bcef", core_1, core_2)
    if paired.shape != (m1, n1, m2, n2):
        raise ValueError(
            f"paired shape が (m1, n1, m2, n2) ではありません: got={tuple(paired.shape)}, expected={(m1, n1, m2, n2)}"
        )
    grouped = paired.permute(0, 2, 1, 3)          # (m1, m2, n1, n2)
    return grouped.reshape(m1 * m2, n1 * n2)


## 7. 演習4 — dense reconstruction を使わない TTLinear direct contraction

入力、

$$
X\in\mathbb{R}^{B\times(n_1n_2)}
$$

を、

$$
\mathcal X\in\mathbb{R}^{B\times n_1\times n_2}
$$

へ reshape します。

求める output tensor は、

$$
\mathcal Y_{b,i_1,i_2}
=
\sum_{j_1,j_2,\alpha}
\mathcal X_{b,j_1,j_2}
G^{(1)}_{1,i_1,j_1,\alpha}
G^{(2)}_{\alpha,i_2,j_2,1}.
$$

shape は、

$$
\mathcal Y\in\mathbb{R}^{B\times m_1\times m_2}.
$$

最後に $(B,m_1m_2)$ へ戻し、bias があれば加えます。


In [30]:
# TODO 4:
def tt_linear_2core_direct(
    x: torch.Tensor,
    core_1: torch.Tensor,
    core_2: torch.Tensor,
    bias: torch.Tensor | None = None,
) -> torch.Tensor:
    """2-core TT-matrix を direct contraction して Linear forward を計算する。"""
#
    # 必須:
    # - shape validation と ValueError
    if x.ndim != 2:
        raise ValueError(f"x は2階である必要があります: shape={tuple(x.shape)}")
    if core_1.ndim != 4:
        raise ValueError(f"core_1 は4階である必要があります: shape={tuple(core_1.shape)}")
    if core_2.ndim != 4:
        raise ValueError(f"core_2 は4階である必要があります: shape={tuple(core_2.shape)}")
    if core_1.shape[0] != 1:
        raise ValueError(f"core_1 の左 boundary rank は1である必要があります: shape={tuple(core_1.shape)}")
    if core_2.shape[-1] != 1:
        raise ValueError(f"core_2 の右 boundary rank は1である必要があります: shape={tuple(core_2.shape)}")
    if core_1.shape[3] != core_2.shape[0]:
        raise ValueError(
            f"internal rank が一致しません: core_1.shape[-1]={core_1.shape[3]}, core_2.shape[0]={core_2.shape[0]}"
        )

    _, m1, n1, _ = core_1.shape
    _, m2, n2, _ = core_2.shape
    if x.shape[1] != n1 * n2:
        raise ValueError(
            f"x.shape[1] が n1*n2 と一致しません: x.shape[1]={x.shape[1]}, n1*n2={n1 * n2}"
        )
    if bias is not None and (bias.ndim != 1 or bias.shape[0] != m1 * m2):
        raise ValueError(
            f"bias は shape ({m1 * m2},) である必要があります: "
            f"got {None if bias is None else tuple(bias.shape)}"
        )
    # - x -> (B, n1, n2)
    x = x.reshape(x.shape[0], n1, n2)
    
    # - TT cores と直接 contraction
    x3 = x.reshape(x.shape[0], n1, n2)          # (B, n1, n2)
    g1 = core_1[0]                               # (m1, n1, r)
    g2 = core_2[..., 0]                          # (r, m2, n2)
    y = torch.einsum("bjk,ijr,rpk->bip", x3, g1, g2)

    # - 出力 shape は (B, m1*m2)
    y = y.reshape(y.shape[0], m1 * m2)  # (B, m1, m2) -> (B, m)

    # - bias は最後に加える
    if bias is not None:
        y = y + bias

    return y


## 8. Toy dense Linear と input batch

ここは中心学習対象ではないため完成済みにします。


In [31]:
dense_linear = nn.Linear(
    input_dim,
    output_dim,
    bias=True,
    dtype=dtype,
    device=device,
)

x = torch.randn(
    batch_size,
    input_dim,
    dtype=dtype,
    device=device,
)

weight_dense = dense_linear.weight.detach().clone()
bias = dense_linear.bias.detach().clone()

assert x.shape == (batch_size, input_dim)
assert weight_dense.shape == (output_dim, input_dim)
assert bias.shape == (output_dim,)

print("x.shape:", tuple(x.shape))
print("weight_dense.shape:", tuple(weight_dense.shape))
print("bias.shape:", tuple(bias.shape))


x.shape: (5, 12)
weight_dense.shape: (20, 12)
bias.shape: (20,)


## 9. 演習5 — TT-rank truncation と weight error

rank $2$ で TT-SVD を行い、$\widetilde W$ を作ります。

$$
\Delta W=\widetilde W-W.
$$

計算するもの：

$$
\|\Delta W\|_F,
\qquad
\|\Delta W\|_2,
\qquad
\frac{\|\Delta W\|_F}{\|W\|_F}.
$$

さらに、

$$
\boxed{\|\Delta W\|_2=\sigma_{\max}(\Delta W)}
$$

と、

$$
\boxed{\|\Delta W\|_2\le\|\Delta W\|_F}
$$

を数値確認します。


In [32]:
from __future__ import annotations

import math


def divisors(value: int, min_divisor: int) -> list[int]:
    """value の約数のうち min_divisor 以上のものを昇順で返す。"""
    if value < 1:
        raise ValueError(f"value must be positive, but got {value}.")
    if min_divisor < 1:
        raise ValueError(
            f"min_divisor must be at least 1, but got {min_divisor}."
        )

    result: list[int] = []

    # sqrt(value) まで見ればペア約数で全部拾える
    for divisor in range(min_divisor, math.isqrt(value) + 1):
        if value % divisor == 0:
            result.append(divisor)

            paired_divisor = value // divisor
            if paired_divisor != divisor:
                result.append(paired_divisor)

    # value 自身も候補に含める（例: 最後の因子）
    if value >= min_divisor:
        result.append(value)

    return sorted(set(result))


def unordered_factorizations(
    value: int,
    num_modes: int,
    min_factor: int = 2,
) -> list[tuple[int, ...]]:
    """value を num_modes 個の非減少な整数因子の積へ分解して返す。

    非減少（a <= b <= c ...）にすることで、
    (2, 6) と (6, 2) のような順序違いを同一視する。
    """
    if value < 1:
        raise ValueError(f"value must be positive, but got {value}.")
    if num_modes < 1:
        raise ValueError(
            f"num_modes must be at least 1, but got {num_modes}."
        )
    if min_factor < 1:
        raise ValueError(
            f"min_factor must be at least 1, but got {min_factor}."
        )

    def recurse(
        remaining: int,
        factors_left: int,
        lower_bound: int,
    ) -> list[tuple[int, ...]]:
        """非減少制約つきで factor tuple を再帰列挙する。

        lower_bound:
            直前までに選んだ因子。これ以上の値だけを次に選ぶ。
        """
        # 最後の1因子は残りそのもの
        if factors_left == 1:
            if remaining >= lower_bound:
                return [(remaining,)]
            return []

        candidates: list[tuple[int, ...]] = []

        for factor in divisors(remaining, lower_bound):
            if factor > remaining:
                continue

            next_remaining = remaining // factor

            # 残り因子をすべて「今の factor 以上」にできないなら枝刈り
            if next_remaining < factor ** (factors_left - 1):
                continue

            for suffix in recurse(
                remaining=next_remaining,
                factors_left=factors_left - 1,
                lower_bound=factor,  # 非減少: 次は factor 以上
            ):
                candidates.append((factor,) + suffix)

        return candidates

    return recurse(
        remaining=value,
        factors_left=num_modes,
        lower_bound=min_factor,
    )


def factor_balance_score(
    factors: tuple[int, ...],
) -> tuple[float, int, int]:
    """小さいほど均等な factorization になる score を返す。

    比較キー（辞書式）:
        1. max/min が小さいほど均等
        2. 同率なら max-min が小さいほど均等
        3. 同率なら最大 factor が小さい方を優先
    """
    if len(factors) == 0:
        raise ValueError("factors must not be empty.")
    if min(factors) <= 0:
        raise ValueError(f"all factors must be positive, but got {factors}.")

    minimum = min(factors)
    maximum = max(factors)

    return (
        maximum / minimum,
        maximum - minimum,
        maximum,
    )


def balanced_modes(
    value: int,
    num_modes: int,
    min_factor: int = 2,
) -> tuple[int, ...]:
    """value を num_modes 個のなるべく均等な整数因子へ自動分解する。

    例:
        balanced_modes(512, 3) → (8, 8, 8)
        balanced_modes(784, 3) → なるべく均等な3因子
    """
    candidates = unordered_factorizations(
        value=value,
        num_modes=num_modes,
        min_factor=min_factor,
    )

    if len(candidates) == 0:
        raise ValueError(
            f"Cannot factorize value={value} into {num_modes} factors "
            f"that are each at least {min_factor}."
        )

    # score が最小の分解を採用
    best = min(candidates, key=factor_balance_score)

    if math.prod(best) != value:
        raise RuntimeError(
            f"Internal error: product of {best} is not {value}."
        )

    return best


In [ ]:
# TODO 5:
#
# 計算:
# delta_w_fro
# delta_w_spectral
# weight_dense_fro
# relative_weight_error
# delta_w_singular_values
m, n = weight_dense.shape
num_tt_cores = 2
rank = 2
m_modes = balanced_modes(m, num_tt_cores)
n_modes = balanced_modes(n, num_tt_cores)
# core_1, core_2 = ...
core_1, core_2 = tt_svd_matrix_2core_with_rank(weight_dense, m_modes, n_modes, rank)
# weight_tt = ...
weight_tt = reconstruct_tt_matrix_2core(core_1, core_2)

# delta_w = weight_tt - weight_dense
delta_w = weight_tt - weight_dense
delta_w_fro = torch.linalg.matrix_norm(delta_w, ord="fro")
delta_w_spectral = torch.linalg.matrix_norm(delta_w, ord=2)
weight_dense_fro = torch.linalg.matrix_norm(weight_dense, ord="fro")
from nn_compression.metrics import relative_frobenius_error

relative_weight_error = relative_frobenius_error(weight_dense, weight_tt)
delta_w_singular_values = torch.linalg.svdvals(delta_w)

# shape assert:
# core_1: (1, m1, n1, tt_rank_truncated)
assert core_1.shape == (1, m_modes[0], n_modes[0], rank)
# core_2: (tt_rank_truncated, m2, n2, 1)
assert core_2.shape == (rank, m_modes[1], n_modes[1], 1)
# weight_tt.shape == weight_dense.shape
assert weight_tt.shape == weight_dense.shape
# delta_w.shape == weight_dense.shape
assert delta_w.shape == weight_dense.shape

# assert_close で
# ||Delta_W||_2 == max singular value
assert_close(
    delta_w_spectral,
    delta_w_singular_values.max(),
    rtol=rtol,
    atol=atol,
)
# assert で
# ||Delta_W||_2 <= ||Delta_W||_F + inequality_atol
assert float(delta_w_spectral) <= float(delta_w_fro) + inequality_atol
# ||Delta_W||_F > 0
assert float(delta_w_fro) > 0


## 10. 演習6 — $\Delta Y=X\Delta W^{\mathsf T}$ の数値検証

まず、direct contraction で計算した $\widetilde Y$ と、検証用に dense reconstruction した $\widetilde W$ を使う、

$$
X\widetilde W^{\mathsf T}+b
$$

が一致することを確認します。

その後、

$$
\Delta Y_{\mathrm{direct}}=\widetilde Y-Y
$$

と、

$$
\Delta Y_{\mathrm{formula}}=X\Delta W^{\mathsf T}
$$

を比較します。

確認したい恒等式は、

$$
\boxed{\Delta Y=\widetilde Y-Y=X\Delta W^{\mathsf T}}.
$$


In [ ]:
# TODO 6:
import torch.nn.functional as F

with torch.no_grad():
    y_dense = F.linear(x, weight_dense, bias)
    y_tt = tt_linear_2core_direct(x, core_1, core_2, bias)
    y_tt_from_reconstructed_weight = F.linear(x, weight_tt, bias)  # X W̃^T + b
#
# assert_close で
# y_tt ≈ y_tt_from_reconstructed_weight
assert_close(y_tt, y_tt_from_reconstructed_weight, rtol=rtol, atol=atol)
#
delta_y_direct = y_tt - y_dense
delta_y_formula = x @ delta_w.T
#
# assert_close で
# delta_y_direct ≈ delta_y_formula
assert_close(delta_y_direct, delta_y_formula, rtol=rtol, atol=atol)

# shape assert:
# y_dense: (batch_size, output_dim)
assert y_dense.shape == (batch_size, output_dim)
# y_tt: (batch_size, output_dim)
assert y_tt.shape == (batch_size, output_dim)
# delta_y_direct: (batch_size, output_dim)
assert delta_y_direct.shape == (batch_size, output_dim)

# delta_y_fro = ||Delta_Y||_F
delta_y_fro = torch.linalg.matrix_norm(
    delta_y_direct,
    ord="fro",
)
# delta_y_fro > 0 を確認
assert float(delta_y_fro) > 0


## 11. 演習7 — 二つの norm 上界を数値検証する

実測値は、

$$
\mathrm{measured\_output\_error}=\|\Delta Y\|_F.
$$

第1上界：

$$
\mathrm{bound}_1=\|X\|_F\|\Delta W\|_2.
$$

第2上界：

$$
\mathrm{bound}_2=\|X\|_2\|\Delta W\|_F.
$$

slack は、

$$
\mathrm{slack}_k=\mathrm{bound}_k-\|\Delta Y\|_F.
$$

ratio は、

$$
\mathrm{ratio}_k=\frac{\|\Delta Y\|_F}{\mathrm{bound}_k}.
$$

正しい上界なら、丸め誤差を除いて $0<\mathrm{ratio}_k\le1$ です。


In [35]:
# TODO 7:
x_fro = torch.linalg.vector_norm(x)
x_spectral = torch.linalg.matrix_norm(x, ord=2)
measured_output_error = delta_y_fro
#
# bound_1 = ||X||_F * ||Delta_W||_2
bound_1 = x_fro * delta_w_spectral
# bound_2 = ||X||_2 * ||Delta_W||_F
bound_2 = x_spectral * delta_w_fro
#
# assert:
# measured_output_error <= bound_1 + inequality_atol
assert float(measured_output_error) <= float(bound_1) + inequality_atol
# measured_output_error <= bound_2 + inequality_atol
assert float(measured_output_error) <= float(bound_2) + inequality_atol
# bound_1 > 0
assert float(bound_1) > 0
# bound_2 > 0
assert float(bound_2) > 0
#
# slack_1, slack_2
slack_1 = bound_1 - measured_output_error
slack_2 = bound_2 - measured_output_error
# ratio_1, ratio_2
ratio_1 = measured_output_error / bound_1
ratio_2 = measured_output_error / bound_2
#
# ratio_1 > 0
assert float(ratio_1) > 0
# ratio_2 > 0
assert float(ratio_2) > 0
# ratio_1 <= 1 + inequality_atol
assert float(ratio_1) <= 1.0 + inequality_atol
# ratio_2 <= 1 + inequality_atol
assert float(ratio_2) <= 1.0 + inequality_atol


## 12. 演習8 — 結果を整理して表示する

次を見やすく表示してください。

- Shapes：$X,W,\widetilde W,\Delta W,Y,\Delta Y$, TT cores
- Weight error：$\|\Delta W\|_F$, $\|\Delta W\|_2$, relative weight error
- Output error：$\|\Delta Y\|_F$
- Upper bounds：2つの bound、slack、ratio
- Identity verification：$\max|\Delta Y_{direct}-\Delta Y_{formula}|$


In [36]:
# TODO 8:
identity_max_abs = (delta_y_direct - delta_y_formula).abs().max()

print("=== Shapes ===")
print(f"X.shape       = {tuple(x.shape)}")
print(f"W.shape       = {tuple(weight_dense.shape)}")
print(f"W_tt.shape    = {tuple(weight_tt.shape)}")
print(f"Delta_W.shape = {tuple(delta_w.shape)}")
print(f"Y.shape       = {tuple(y_dense.shape)}")
print(f"Delta_Y.shape = {tuple(delta_y_direct.shape)}")

print("\n=== TT-matrix cores ===")
print(f"core_1.shape = {tuple(core_1.shape)}")
print(f"core_2.shape = {tuple(core_2.shape)}")

print("\n=== Weight error ===")
print(f"||Delta_W||_F          = {float(delta_w_fro):.6e}")
print(f"||Delta_W||_2          = {float(delta_w_spectral):.6e}")
print(f"relative_weight_error  = {float(relative_weight_error):.6e}")

print("\n=== Output error ===")
print(f"||Delta_Y||_F = {float(delta_y_fro):.6e}")

print("\n=== Norm upper bounds ===")
print(f"bound_1 = ||X||_F * ||Delta_W||_2 = {float(bound_1):.6e}")
print(f"bound_2 = ||X||_2 * ||Delta_W||_F = {float(bound_2):.6e}")
print(f"slack_1 = {float(slack_1):.6e}")
print(f"slack_2 = {float(slack_2):.6e}")
print(f"ratio_1 = {float(ratio_1):.6e}")
print(f"ratio_2 = {float(ratio_2):.6e}")

print("\n=== Identity verification ===")
print(f"max|Delta_Y_direct - Delta_Y_formula| = {float(identity_max_abs):.6e}")

print("\nAll numerical checks passed.")


=== Shapes ===
X.shape       = (5, 12)
W.shape       = (20, 12)
W_tt.shape    = (20, 12)
Delta_W.shape = (20, 12)
Y.shape       = (5, 20)
Delta_Y.shape = (5, 20)

=== TT-matrix cores ===
core_1.shape = (1, 4, 3, 2)
core_2.shape = (2, 5, 4, 1)

=== Weight error ===
||Delta_W||_F          = 2.010011e+00
||Delta_W||_2          = 9.661027e-01
relative_weight_error  = 8.047474e-01

=== Output error ===
||Delta_Y||_F = 4.814268e+00

=== Norm upper bounds ===
bound_1 = ||X||_F * ||Delta_W||_2 = 8.592132e+00
bound_2 = ||X||_2 * ||Delta_W||_F = 1.171245e+01
slack_1 = 3.777864e+00
slack_2 = 6.898180e+00
ratio_1 = 5.603112e-01
ratio_2 = 4.110386e-01

=== Identity verification ===
max|Delta_Y_direct - Delta_Y_formula| = 4.440892e-16

All numerical checks passed.


## 13. 解釈と結論

### $\|\Delta W\|_F$

$$
\|\Delta W\|_F^2=\sum_{i,j}(\Delta W_{ij})^2.
$$

weight 全要素の総二乗誤差です。

### $\|\Delta W\|_2$

$$
\|\Delta W\|_2=\sigma_{\max}(\Delta W)
$$

であり、

$$
\|\Delta Wv\|_2\le\|\Delta W\|_2\|v\|_2.
$$

weight error が入力方向をどれだけ増幅し得るかを表します。

### $\|\Delta Y\|_F$

$$
\Delta Y=X\Delta W^{\mathsf T}
$$

なので、$\|\Delta Y\|_F$ は **この特定の input batch $X$ に対して実際に観測された layer output error** です。

### 上界は一般に等号ではない

$$
\|\Delta Y\|_F\le\|X\|_F\|\Delta W\|_2,
$$

$$
\|\Delta Y\|_F\le\|X\|_2\|\Delta W\|_F
$$

は一般に保守的になり得ます。

また、どちらの上界が小さいかは $X$ と $\Delta W$ の特異値分布に依存します。

weight error が小さくても、実際の output error は入力 batch に依存します。

$$
\boxed{\Delta Y=X\Delta W^{\mathsf T}}.
$$

今回扱ったのは、

$$
\boxed{
\text{TT-rank truncation}
\rightarrow\Delta W
\rightarrow\Delta Y
\rightarrow\text{Linear output norm bound}
}
$$

までです。

次のテーマである、

$$
\|\widetilde H-H\|_F\le\|\widetilde Z_1-Z_1\|_F
$$

すなわち ReLU の 1-Lipschitz 性による hidden activation error の評価には、この Notebook では進みません。
